# Parameters

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd

# ===== CONFIGURAÇÃO DE CAMINHOS =====
current_notebook = Path.cwd()  
project_root = current_notebook.parent.parent 

# Adiciona o diretório raiz ao sys.path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Adiciona o diretório Modules ao sys.path
modules_dir = project_root / "Modules"
if str(modules_dir) not in sys.path:
    sys.path.insert(0, str(modules_dir))

# ===== IMPORTS DOS MÓDULOS =====
import Modules.ClusterSHADEModule as cluster
import Modules.FutureAnalysisModule as fa
from Modules.config import CONFIG

# ===== CONFIGURAÇÕES DO PROJETO =====
DATAPATH = CONFIG["datapath"]
COVID_TRAIN_DATA_FILE = CONFIG["covid_train_data_file"]
COVID_TEST_DATA_FILE = CONFIG["covid_test_data_file"]
FUTURE_DATA_FILE = CONFIG["future_data_file"]

CONTROL_GROUP_TRAIN = CONFIG["control_group_train"]
CONTROL_GROUP_TEST = CONFIG["control_group_test"]
CONTROL_GROUP_READMISSION = CONFIG["control_group_readmission"]

FIGSIZE_CLUSTER_HEATMAP = CONFIG["figsize_cluster_heatmap"]
FIGSIZE_FUTURE_HEATMAP = CONFIG["figsize_future_heatmap"]
IMAGES_SAVE_PATH = CONFIG["image_save_path"]
TRIALS_OPTUNA = 250

# Import data

In [ ]:
# ===== CARREGAMENTO DOS DADOS =====
data_folder = current_notebook / DATAPATH

covid_train = pd.read_csv(data_folder / COVID_TRAIN_DATA_FILE)
covid_test = pd.read_csv(data_folder / COVID_TEST_DATA_FILE)
future_data = pd.read_csv(data_folder / FUTURE_DATA_FILE)

control_train = pd.read_csv(data_folder / CONTROL_GROUP_TRAIN)
control_test = pd.read_csv(data_folder / CONTROL_GROUP_TEST)
control = pd.concat([control_train, control_test], axis=0)
control_readmission = pd.read_csv(data_folder / CONTROL_GROUP_READMISSION)

# Feature engineering: morte após a internação
covid_train['died_after'] = ((covid_train['died'] == 1) & (covid_train['died_in_stay'] == 0)).astype(int)
covid_test['died_after'] = ((covid_test['died'] == 1) & (covid_test['died_in_stay'] == 0)).astype(int)
future_data['died_after'] = ((future_data['died'] == 1) & (future_data['died_in_stay'] == 0)).astype(int)

In [ ]:
data_covid = pd.concat([covid_train, covid_test], axis=0)
data_covid = data_covid.sample(frac=1, random_state=42).reset_index(drop=True)

# Mice Data

In [ ]:
categorical_features = [
            "myocardial_infarct",
            "congestive_heart_failure",
            "peripheral_vascular_disease",
            "cerebrovascular_disease",
            "dementia",
            "chronic_pulmonary_disease",
            "rheumatic_disease",
            "peptic_ulcer_disease",
            "mild_liver_disease",
            "diabetes_without_cc",
            "diabetes_with_cc",
            "paraplegia",
            "renal_disease",
            "malignant_cancer",
            "severe_liver_disease",
            "metastatic_solid_tumor",
            "aids",
            "gender_M",
            "died_in_stay",
            "died_after",
            "died",
            "COVID"
        ]

In [ ]:
features_not_considered = ['died', 'died_in_stay', 'died_after', 'COVID', 'subject_id', 'hadm_id']

In [ ]:
helper = cluster.SHADEClusterHelper(data=data_covid, features_not_considered=features_not_considered)

## Find best hyperparameters for SHADE

In [ ]:
param = {
    "batch_size": [64, 128, 256, 500],
    "clustering_epochs": [100, 150, 200],
    "clustering_lr": {"min": 1e-4, "max": 1e-2},
}

### DBCV

In [ ]:
os.environ["PYTHONWARNINGS"] = "ignore"
dbcv_df, dbcv_param, dbcv_best = helper.optuna_grid_search(
    parameters=param, 
    n_trials=TRIALS_OPTUNA, 
    save_storage=True, 
    metric="dbcv",
)

In [ ]:
dbcv_param = {'batch_size': 64, 'clustering_epochs': 100, 'clustering_lr': 0.003890211602331156}

### DISCO

In [ ]:
os.environ['PYTHONWARNINGS'] = 'ignore'
disco_df, disco_param, disco_best = helper.optuna_grid_search(
    parameters=param, 
    n_trials=TRIALS_OPTUNA, 
    save_storage=True, 
    metric="disco",
)

In [ ]:
disco_param = {'batch_size': 500, 'clustering_epochs': 100, 'clustering_lr': 0.006192247528471769}

### DSI

In [ ]:
os.environ['PYTHONWARNINGS'] = 'ignore'
dsi_df, dsi_param, dsi_best = helper.optuna_grid_search(
    parameters=param, 
    n_trials=TRIALS_OPTUNA, 
    save_storage=True, 
    metric="dsi",
)

In [ ]:
dsi_param = {'batch_size': 500, 'clustering_epochs': 150, 'clustering_lr': 0.008008198436051767}

### Silhouette

In [ ]:
os.environ['PYTHONWARNINGS'] = 'ignore'
silhouette_df, silhouette_param, silhouette_best = helper.optuna_grid_search(
    parameters=param, 
    n_trials=TRIALS_OPTUNA, 
    save_storage=True, 
    metric="silhouette",
)

In [ ]:
silhouette_param = {'batch_size': 64, 'clustering_epochs': 100, 'clustering_lr': 0.004225424722098411}

### Metrics

In [ ]:
helper.clustering(
    batch_size=dbcv_param["batch_size"],
    clustering_epochs=dbcv_param["clustering_epochs"],
    clustering_optimizer_params={"lr": dbcv_param["clustering_lr"]},
)
helper.get_metrics()

In [ ]:
helper.clustering(
    batch_size=disco_param["batch_size"],
    clustering_epochs=disco_param["clustering_epochs"],
    clustering_optimizer_params={"lr": disco_param["clustering_lr"]},
)
helper.get_metrics()

In [ ]:
helper.clustering(
    batch_size=dsi_param["batch_size"],
    clustering_epochs=dsi_param["clustering_epochs"],
    clustering_optimizer_params={"lr": dsi_param["clustering_lr"]},
)
helper.get_metrics()

In [ ]:
helper.clustering(
    batch_size=silhouette_param["batch_size"],
    clustering_epochs=silhouette_param["clustering_epochs"],
    clustering_optimizer_params={"lr": silhouette_param["clustering_lr"]},
)
helper.get_metrics()

## Best Result - All

In [ ]:
best_param = silhouette_param

In [ ]:
helper.clustering(
    batch_size=best_param["batch_size"],
    clustering_epochs=best_param["clustering_epochs"],
    clustering_optimizer_params={"lr": best_param["clustering_lr"]},
)
helper.get_metrics()

In [ ]:
helper.heatmap_clusters_categorical(
    figsize=FIGSIZE_CLUSTER_HEATMAP,
    # savepath=IMAGES_SAVE_PATH + "shade-all-categorical"
    relative_total=True
)

In [ ]:
helper.show_cluster_compare_numerical(
    figsize=(10, 15),
    # savepath=IMAGES_SAVE_PATH + "shade-all-numerical"
)

In [ ]:
selected_clusters = [-1, 0]

In [ ]:
helper.set_clustered_autoencoder()

In [ ]:
helper.show_clustered_autoencoder(
    selected_clusters=selected_clusters,
    # savepath=IMAGES_SAVE_PATH + "shade-autoencoder-all",
)

##### Future data


In [ ]:
future_helper = fa.FutureAnalysisHelper(
    helper.clustered_data, future_data, control, control_readmission
)
delta = future_helper.get_delta_clusters(percentage=True, relative_total=True)
future_helper.show_delta_heatmap(
    figsize=FIGSIZE_FUTURE_HEATMAP,
    relative_total=True,
    selected_clusters=selected_clusters,
    savepath=IMAGES_SAVE_PATH + "shade-all-future",
)

In [ ]:
future_helper.get_mean_readmission()

In [ ]:
future_helper.get_mean_days_gap()

In [ ]:
future_helper.get_mortality_rates(only_first_admission=True)

### Add Logs

In [ ]:
log_file = "../metrics.csv"
current_dir = os.getcwd()
log_file_path = os.path.join(current_dir, log_file)

metrics = helper.get_metrics()

# Add line to save log
if os.path.exists(log_file_path):
    with open(log_file_path, 'a') as f:
        f.write(f"Shade, None, Comprehensive, {metrics['disco']}, {metrics['dbcv']}, {metrics['dsi']}, {metrics['silhouette']}\n")